## Notebook Description

The notebook will contain 5 main components:
- Configuration
- Data collection & cleaning
- Feature creation
- Model training
- Results

Each main component may contain mutiple subcomponents depending on the complexity of the component
The novelty added in this notebook is the usage of three new features:
- Quali Position
- Historical influence of quali result to the final race position on each track
- FP2 pace

Disclaimer: This is a notebook that uses as a starting point the Austin_GP_Winner_Predictor made by Mariana Antaya


### Step 0 - Package Installation

In [8]:
# %pip install fastf1 scikit-learn xgboost matplotlib

### Step 1 - Configuration

- Import the necessary libraries 
- Configure different necessities as cache

In [9]:
from pathlib import Path
from datetime import datetime, timezone
from collections import deque, defaultdict
import pandas as pd
import numpy as np
import time
import fastf1
from fastf1.ergast import Ergast
import sklearn
#from sklearn.preprocessing import PolynomialFeatures
import matplotlib.pyplot as plt

cache_dir = Path('data_cache')
cache_dir.mkdir(exist_ok=True)
fastf1.Cache.enable_cache(str(cache_dir))

api = Ergast(result_type='pandas', limit=1000)

YEAR = 2026
START_YEAR = 2012
AUSTIN_ID = 'americas'
MAX_SLOT = 20            # starting slots beyond this are merged into the last slot

### Step 2 - Data Collection

This step will be split into two substeps:
- Step 2.1 Data Collection & Cleaning as in reference notebook
- Step 2.2 Data Collection & Cleaning for new features
- Step 2.3 Final Data Aggregation

### Step 2.0 Creating cleaning functions

In [10]:
def clean_results(df, max_slot=MAX_SLOT):
    df = df.copy()
    df['raceDate'] = pd.to_datetime(df['raceDate'])
    for col in ['season', 'round', 'grid', 'position']:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    unusable = df['position'].isna() | df['grid'].isna()
    if unusable.any():
        print(f"Dropping {unusable.sum()} rows without a usable grid/position")
    df = df.loc[~unusable].copy()
    df[['season', 'round', 'grid', 'position']] = (
        df[['season', 'round', 'grid', 'position']].astype(int))

    # Starting slot = actual grid slot (penalties already included).
    # grid == 0 means pit-lane start -> placed behind the last gridded car.
    df['pit_lane_start'] = df['grid'].eq(0)
    last_gridded = (df['grid'].where(df['grid'] > 0)
                      .groupby([df['season'], df['round']]).transform('max')
                      .fillna(max_slot))
    df['starting_slot'] = np.where(df['pit_lane_start'], last_gridded + 1, df['grid'])
    df['starting_slot'] = df['starting_slot'].clip(upper=max_slot).astype(int)

    df['is_win'] = df['position'].eq(1)
    df['is_podium'] = df['position'].le(3)
    df['is_top10'] = df['position'].le(10)
    df['positions_gained'] = df['starting_slot'] - df['position']

    return df.sort_values(['raceDate', 'round', 'position']).reset_index(drop=True)


def check_races(df):
    """Returns races with a data problem (should be an empty frame)."""
    per_race = df.groupby(['season', 'round']).agg(
        circuitId=('circuitId', 'first'),
        rows=('driverId', 'size'),
        unique_drivers=('driverId', 'nunique'),
        winners=('is_win', 'sum'),
        p1_starters=('starting_slot', lambda s: (s == 1).sum()),
    )
    return per_race[(per_race['winners'] != 1)
                    | (per_race['rows'] != per_race['unique_drivers'])
                    | (per_race['p1_starters'] != 1)]


#### Step 2.1 Data Collection - For both processing types

In [11]:
#Define the variables of interest coming from both metadata and the real data
META_COLS = ['season', 'round', 'raceDate', 'circuitId', 'raceName']
KEEP_COLS = META_COLS + ['driverId', 'givenName', 'familyName',
                         'constructorId', 'grid', 'position',
                         'positionText', 'status', 'points',
                         'season', 'round']

#Create the function that will retrieve the data from the api
def fetch_race_results(cache_name, refresh=False, **query):
    """Download race results for any Ergast filter (season=..., circuit=...).
    Cached in data_cache/<cache_name>.csv; pass refresh=True to re-download."""
    csv_path = cache_dir / f"{cache_name}.csv"
    if csv_path.exists() and not refresh:
        return pd.read_csv(csv_path, parse_dates=['raceDate'])

    page = api.get_race_results(**query)
    chunks = []
    while True:
        for (_, meta), result in zip(page.description.iterrows(), page.content):
            frame = pd.DataFrame(result).copy()
            for col in META_COLS:
                frame[col] = meta[col]
            frame['raceDate'] = pd.Timestamp(meta['raceDate']).tz_localize(None).normalize()
            chunks.append(frame)
        try:
            time.sleep(0.4)
            page = page.get_next_result_page()
        except ValueError as exc:
            if 'No more data' in str(exc):
                break
            raise

    if not chunks:
        return pd.DataFrame()

    frame = pd.concat(chunks, ignore_index=True)
    frame = frame.drop_duplicates(['season', 'round', 'driverId'])
    # frame = frame[[c for c in KEEP_COLS if c in frame.columns]]
    frame.to_csv(csv_path, index=False)
    return frame

#Store all retrieved data in a single df/csv. Later it will be used differently based on the analysis purpose
schedule = api.get_race_schedule(season=YEAR)
austin = schedule.loc[schedule['circuitId'].eq('americas')]
if len(austin) != 1:
    raise ValueError("No unique Austin race in this season schedule. Check target year")

target = austin.iloc[0]
target_date = pd.Timestamp(target['raceDate']).tz_localize(None).normalize()

frames = []
for year in range(START_YEAR, YEAR):
    part = fetch_race_results(f"season_{year}", season = year)
    if not part.empty:
        frames.append(part)
    print(f"{year}: {len(part)} driver results loaded")

raw_all = pd.concat(frames, ignore_index=True)
raw_all['raceDate'] = pd.to_datetime(raw_all['raceDate']).dt.tz_localize(None)
today = pd.Timestamp(datetime.now(timezone.utc).date())

#Create DF to be used in recent form features calculation
recent_form_df = raw_all.loc[
    (raw_all['raceDate'] < target_date) & 
    (raw_all['raceDate'] < today)
].copy()
recent_form_df['position'] = pd.to_numeric(recent_form_df['position'], errors='raise')
recent_form_df['points'] = pd.to_numeric(recent_form_df['points'], errors='raise')
recent_form_df = recent_form_df.sort_values(['raceDate', 'season', 'round', 'position'])

recent_form_df.head()

#Create DF to be used in starting slot analysis
cleaned_df = clean_results(raw_all)

starting_slot_df = cleaned_df.loc[
    (cleaned_df['raceDate'] < target_date) &
    (cleaned_df['raceDate'] < today)
].copy()





2012: 478 driver results loaded
2013: 418 driver results loaded
2014: 407 driver results loaded
2015: 378 driver results loaded
2016: 462 driver results loaded
2017: 400 driver results loaded
2018: 420 driver results loaded
2019: 420 driver results loaded
2020: 340 driver results loaded
2021: 440 driver results loaded
2022: 440 driver results loaded
2023: 440 driver results loaded
2024: 479 driver results loaded
2025: 479 driver results loaded


### Step 2.3 Features creation for driver recent form

In [12]:
FEATURES = ['driver_finish_5', 'driver_points_5', 'driver_win_rate_5',
            'driver_podium_rate_5', 'team_points_5', 'team_win_rate_5',
            'circuit_finish_3', 'circuit_win_rate_3', 'driver_recent_form_count']

drivers = defaultdict(lambda: deque(maxlen=5))
teams = defaultdict(lambda: deque(maxlen=5))
circuits = defaultdict(lambda: deque(maxlen=3))

def avg(records, column):
    return float(np.mean([r[column] for r in records])) if records else np.nan

def features(driver, team, circuit):
    d, t, c = drivers[driver], teams[team], circuits[(driver, circuit)]
    return dict(zip(FEATURES, [avg(d, 'finish'), avg(d, 'points'), avg(d, 'win'),
                    avg(d, 'podium'), avg(t, 'points'), avg(t, 'win'), avg(c, 'finish'),
                    avg(c, 'win'), len(d)]))

rows = []

for (season, rnd), race in recent_form_df.groupby(['season', 'round'], sort=True):
    for _, r in race.iterrows():
        rows.append({**features(r.driverId, r.constructorId, r.circuitId),
                     'season': int(season), 'round':int(rnd),
                     'race_id': f'{season}-{rnd}', 'raceName': r.raceName,
                     'circuitId': r.circuitId, 'driverId': r.driverId,
                     'driver':f'{r.givenName} {r.familyName}', 'won':int(r.position == 1)})
    for _, r in race.iterrows():
        record = dict(finish=float(r.position), points=float(r.points),
                      win=int(r.position == 1), podium=int(r.position <= 3))
        drivers[r.driverId].append(record)
        circuits[(r.driverId, r.circuitId)].append(record)
    for team, team_race in race.groupby('constructorId'):
        teams[team].append(dict(points=float(team_race.points.sum()),
                                win=int(team_race.position.eq(1).any())))

data = pd.DataFrame(rows)
data = data.loc[data.season > START_YEAR].copy()

austin_examples = data.loc[data.circuitId.eq('americas')]

austin_examples.head()

preview = austin_examples.loc[austin_examples.season.eq(austin_examples.season.max())].head(5)
print('Example: inputs available BEFORE the', int(preview.season.iloc[0]), 'Austin race')
display(preview[['driver','driver_finish_5','team_points_5','circuit_finish_3','won']]
        .rename(columns={'driver':'Driver','driver_finish_5':'Recent avg finish',
                         'team_points_5':'Team avg points','circuit_finish_3':'Austin avg finish',
                         'won':'Won the race?'})
        .replace({'Won the race?':{0:'No',1:'Yes'}}).round(1).reset_index(drop=True))
print('This preview shows 3 of the 9 inputs. Lower average finish is better; NaN means no history.')

Example: inputs available BEFORE the 2025 Austin race


,Driver,Recent avg finish,Team avg points,Austin avg finish,Won the race?
0,Max Verstappen,3.0,19.6,1.7,Yes
1,Lando Norris,6.2,26.8,4.0,No
2,Charles Leclerc,8.4,10.0,8.0,No
3,Lewis Hamilton,10.8,10.0,13.7,No
4,Oscar Piastri,6.0,26.8,11.0,No


This preview shows 3 of the 9 inputs. Lower average finish is better; NaN means no history.


### Step 2.5 Features Creation - Starting Slot Influence

In [13]:
austin_hist = starting_slot_df.loc[
    starting_slot_df['circuitId'].eq(AUSTIN_ID)
].copy()

def slot_table(df):
    """"Outcome statistics per starting slot, using every driver of every race."""

    table = df.groupby('starting_slot').agg(
        starts = ('driverId', 'size'),
        wins = ('is_win', 'sum'),
        podiums = ('is_podium', 'sum'),
        mean_finish = ('position', 'mean'),
        mean_gain = ('positions_gained', 'mean')
    )

    table['win_rate'] = table['wins'] / table['starts']
    table['podium_rate'] = table['podiums'] / table['starts']

    return table.round(3)

def race_summary(df):
    """One row per race: where the winner started and how strongly grid order predicted finishing order."""
    rows = []
    for (season, rnd), g in df.groupby(['season', 'round']):
        winner = g.loc[g['is_win'], 'starting_slot']
        rows.append({
            'season': season,
            'round': rnd,
            'raceDate': g['raceDate'].iloc[0],
            'circuitId': g['circuitId'].iloc[0],
            'winner_slot': winner.iloc[0] if len(winner) else np.nan,
            'grid_finish_corr': g['starting_slot'].rank().corr(g['position'].rank()),
        })
    return pd.DataFrame(rows)


def circuit_summary(races):
    return (races.groupby('circuitId')
                 .agg(n_races=('round', 'size'),
                      pole_win_rate=('winner_slot', lambda s: (s == 1).mean()),
                      mean_winner_slot=('winner_slot', 'mean'),
                      grid_finish_corr=('grid_finish_corr', 'mean'))
                 .sort_values('grid_finish_corr', ascending=False)
                 .round(3))


races_hist = race_summary(starting_slot_df)
circuits_hist = circuit_summary(races_hist)

print(circuits_hist.loc[[AUSTIN_ID]])   # Austin's circuit-level numbers
slot_table(austin_hist) 

           n_races  pole_win_rate  mean_winner_slot  grid_finish_corr
circuitId                                                            
americas        13          0.462               2.0             0.519


,starts,wins,podiums,mean_finish,mean_gain,win_rate,podium_rate
starting_slot,,,,,,,
1,13,6,10,4.538,-3.538,0.462,0.769
2,13,5,12,3.154,-1.154,0.385,0.923
3,13,0,7,6.538,-3.538,0.000,0.538
4,13,1,2,8.154,-4.154,0.077,0.154
5,13,0,3,7.308,-2.308,0.000,0.231
6,13,1,1,7.385,-1.385,0.077,0.077
7,13,0,1,10.538,-3.538,0.000,0.077
8,13,0,0,12.846,-4.846,0.000,0.000
9,13,0,0,9.462,-0.462,0.000,0.000


### Step 2.3 Final Data Aggregation

In [21]:
SLOT_FEATURES = ['starting_slot', 'slot_win_prob', 'circuit_grid_corr', 'circuit_prev_races']
ALL_FEATURES = FEATURES + SLOT_FEATURES
KEYS = ['season', 'round', 'raceDate', 'race_id', 'raceName', 'circuitId',
        'driverId', 'driver', 'constructorId']

PRIOR_MEAN_SLOT = 2.5     # assumed average winner slot before a circuit has history
PRIOR_WEIGHT = 2          # how many pseudo-races that assumption counts for

def build_target_rows():
    ran = cleaned_df.loc[
        (cleaned_df['raceDate'] == target_date) & 
        (cleaned_df['circuitId'] == AUSTIN_ID)
    ]

    if len(ran):
        base = pd.DataFrame({
            'driverId': ran['driverId'].to_numpy(),
            'constructorId': ran['constructorId'].to_numpy(),
            'driver': (ran['givenName'] + ' ' + ran['familyName']).to_numpy(),
            'starting_slot': ran['starting_slot'].to_numpy(),
            'won': ran['is_win'].astype(int).to_numpy()
        })
        source = 'race results (official starting grid)'
    else:
        quali = api.get_qualifying_results(season=YEAR, round=int(target['round'])).content[0]
        base = pd.DataFrame({
            'driverId': quali['driverId'].to_numpy(),
            'constructorId': quali['constructorId'].to_numpy(),
            'driver': (quali['givenName'] + ' ' + quali['familyName']).to_numpy(),
            'starting_slot': pd.to_numeric(quali['position']).clip(upper=MAX_SLOT).astype(int).to_numpy(),
            'won': np.nan})
        source = 'qualifying order (grid penalties NOT included)'

    feats = pd.DataFrame([features(r.driverId, r.constructorId, AUSTIN_ID)
                          for r in base.itertuples()])
    out = pd.concat([base.reset_index(drop=True), feats], axis=1)
    out['season'] = YEAR
    out['round'] = int(target['round'])
    out['race_id'] = f"{YEAR}-{int(target['round'])}"
    out['raceName'] = target['raceName']
    out['circuitId'] = AUSTIN_ID
    out['raceDate'] = target_date
    print(f"Target race rows: {len(out)} drivers, grid taken from {source}")

    return out

def prior_sums(values, groups):
    valid = values.notna().astype(int)
    filled = values.fillna(0)
    sum_prev = filled.groupby(groups).cumsum() - filled
    n_prev = valid.groupby(groups).cumsum() - valid
    return sum_prev, n_prev

def slot_priors(races):
    races = races.sort_values(['raceDate', 'round']).reset_index(drop=True)
    ws_sum, ws_n = prior_sums(races['winner_slot'], races['circuitId'])
    corr_sum, corr_n = prior_sums(races['grid_finish_corr'], races['circuitId'])
    mean_slot = (ws_sum + PRIOR_WEIGHT * PRIOR_MEAN_SLOT) / (ws_n + PRIOR_WEIGHT)
    races['q'] = 1 - 1 / mean_slot                    # decay of the geometric slot curve
    races['circuit_prev_races'] = ws_n
    races['circuit_grid_corr'] = corr_sum / corr_n.replace(0, np.nan)
    return races


races = race_summary(starting_slot_df)
target_maker = pd.DataFrame(
    [{
        'season': YEAR,
        'round': int(target['round']),
        'raceDate': target_date,
        'circuitId': AUSTIN_ID,
        'winner_slot': np.nan,
        'grid_finish_corr': np.nan
    }])

races = slot_priors(pd.concat([races, target_maker], ignore_index=True))

# attach the starting slot to your existing feature rows
train_rows = data.merge(
    cleaned_df[['season', 'round', 'driverId', 'raceDate', 'constructorId', 'starting_slot']],
    on=['season', 'round', 'driverId'],
    how='left'
)

lost = train_rows['starting_slot'].isna().sum()

if lost:
    print(f'Dropping {lost} rows without a starting slot')

train_rows = train_rows.dropna(subset=['starting_slot']).copy()
train_rows['starting_slot'] = train_rows['starting_slot'].astype(int)

all_rows = pd.concat([train_rows, build_target_rows()], ignore_index=True)
all_rows = all_rows.merge(races[['season', 'round', 'q', 'circuit_prev_races', 'circuit_grid_corr']],
                          on=['season', 'round'],
                          how='left')

q = all_rows['q'].to_numpy(dtype='float')
slot = all_rows['starting_slot'].to_numpy()
norm = (1 - q ** MAX_SLOT)/(1 - q)
all_rows['slot_win_prob'] = q ** (slot - 1)/norm 

model_table = (all_rows[KEYS + ['won'] + ALL_FEATURES]
               .sort_values(['raceDate', 'round', 'driverId']).reset_index(drop=True))

is_target = model_table['raceDate'].eq(target_date) & model_table['circuitId'].eq(AUSTIN_ID)
train = model_table.loc[~is_target]
target_rows = model_table.loc[is_target]

print(model_table.shape, train.shape, target_rows.shape)

Target race rows: 22 drivers, grid taken from qualifying order (grid penalties NOT included)
(5545, 23) (5523, 23) (22, 23)


### Step 3 Features Extraction and Creation

### Step 4 Model Creation

### Step 5 Results